In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import torch.utils.data as data
import random
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.fucntions import validate_agg_data, adf_test, Evaluator,rmse

In [2]:
Counties = pd.read_csv('../data/interim/Media_per_county.csv')

In [3]:
acols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count']
tcols=['min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']
mcols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count', 'min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']

In [4]:
feature_sets= {'all_media':mcols,
               'articles':acols,
               'twitter':tcols}

features=[mcols,acols,tcols]

target='rate_per_100k'

horizons= [1,3,6]

In [5]:
Counties = Counties.sort_values(
    ['County', 'year_month']
).copy()

county_data = {
    county: group.sort_values('year_month').reset_index(drop=True)
    for county, group in Counties.groupby('County')
}

In [6]:
county_model_data = {}

for county, df in county_data.items():
    df = df.sort_values('year_month').copy()

    df['log_rate'] = np.log1p(df[target])
    df['logdiff_rate'] = df['log_rate'].diff()

    df = df.dropna(subset=['logdiff_rate']).reset_index(drop=True)

    county_model_data[county] = {
        'df': df,
        'y': df['logdiff_rate'],
        'dates': df['year_month'],
        'X': {
            name: df[cols]
            for name, cols in feature_sets.items()
        }
    }

In [7]:
county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = {
                "X_train": X.iloc[:-horizon],
                "X_test": X.iloc[-horizon:],
                "y_train": y.iloc[:-horizon],
                "y_test": y.iloc[-horizon:]
            }

In [8]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train, len(X) - horizon + 1, step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_train": y.iloc[:train_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits

initial_train = 120

county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = (
                walk_forward_splits(
                    X,
                    y,
                    initial_train=initial_train,
                    horizon=horizon,
                    step=1
                )
            )

In [9]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

In [ ]:
XGB_results = {}
XGB_best_params = {}
XGB_importance_rows = []

for county, feature_sets_for_county in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Tune using the first training window
        initial_splits = horizons_for_feature[1]

        if not initial_splits:
            print(f"Skipping {county} / {feature_name}: insufficient data")
            continue

        first_split = initial_splits[0]

        X_initial = first_split["X_train"]
        y_initial = first_split["y_train"].to_numpy().ravel()

        # Hyperparameter search using chronological CV
        search = RandomizedSearchCV(
            estimator=XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                tree_method="hist",
                random_state=42,
                n_jobs=1
            ),
            param_distributions=param_distributions,
            n_iter=12,
            scoring="neg_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            random_state=42,
            n_jobs=-1,
            verbose=0
        )

        search.fit(X_initial, y_initial)

        best_params = search.best_params_
        XGB_best_params[(county, feature_name)] = best_params

        print(f"Tuned {county} | {feature_name}")

        # Evaluate every horizon
        for horizon, splits_for_horizon in horizons_for_feature.items():

            forecasts = []

            for i, split in enumerate(splits_for_horizon):

                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"].to_numpy().ravel()
                y_test = split["y_test"].to_numpy().ravel()

                model = XGBRegressor(
                    **best_params,
                    objective="reg:squarederror",
                    eval_metric="rmse",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=1
                )

                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                # Store predictions
                for lead, (actual, predicted) in enumerate(
                    zip(y_test, predictions), start=1
                ):
                    forecasts.append({
                        "split": i,
                        "lead": lead,
                        "actual": actual,
                        "predicted": predicted
                    })

                # Store feature importance for this fitted split
                for feature, importance in zip(
                    X_train.columns,
                    model.feature_importances_
                ):
                    XGB_importance_rows.append({
                        "County": county,
                        "Feature_Set": feature_name,
                        "Horizon": horizon,
                        "Split": i,
                        "Feature": feature,
                        "Importance": importance
                    })

            XGB_results[(county, feature_name, horizon)] = (
                pd.DataFrame(forecasts)
            )

            print(
                f"{county} | {feature_name} | horizon={horizon} | "
                f"{len(splits_for_horizon)} splits complete"
            )


Tuned Alameda | all_media
Alameda | all_media | horizon=1 | 35 splits complete
Alameda | all_media | horizon=3 | 33 splits complete
Alameda | all_media | horizon=6 | 30 splits complete
Tuned Alameda | articles
Alameda | articles | horizon=1 | 35 splits complete
Alameda | articles | horizon=3 | 33 splits complete
Alameda | articles | horizon=6 | 30 splits complete
Tuned Alameda | twitter
Alameda | twitter | horizon=1 | 35 splits complete
Alameda | twitter | horizon=3 | 33 splits complete
Alameda | twitter | horizon=6 | 30 splits complete
Tuned Contra Costa | all_media
Contra Costa | all_media | horizon=1 | 35 splits complete
Contra Costa | all_media | horizon=3 | 33 splits complete
Contra Costa | all_media | horizon=6 | 30 splits complete
Tuned Contra Costa | articles
Contra Costa | articles | horizon=1 | 35 splits complete
Contra Costa | articles | horizon=3 | 33 splits complete
Contra Costa | articles | horizon=6 | 30 splits complete
Tuned Contra Costa | twitter
Contra Costa | twitter

In [12]:
XGB_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in XGB_results.items()
    ],
    ignore_index=True
)

XGB_predictions_df.to_csv(
    "../data/interim/xgboost_predictions.csv",
    index=False
)

pd.DataFrame(XGB_importance_rows).to_csv(
    "../data/interim/xgboost_feature_importance.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in XGB_best_params.items()
]).to_csv(
    "../data/interim/xgboost_best_params.csv",
    index=False
)

In [14]:
evaluation_rows = []

for (county, feature_name, horizon), df in XGB_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('xgboost_Media_Evals.csv', index=False)